In [1]:
import sys
import os
import torch
sys.path.append(os.path.abspath('../'))

from pathlib import Path
from tqdm import tqdm

# Add both project root and src/ to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))  # <-- add this

from src.utils import load_config
from src.validation import get_tpcf
from src.fm import FlowMatching
from src.egnn import EGNN
from src.pbc_config import min_image, wrap, BOX
from scipy.optimize import linear_sum_assignment

import numpy as np
import pandas as pd 
import torch.nn as nn
import matplotlib.pyplot as plt

# from pathlib import Path
# import sys, os

/home/bartb/venvs/boids/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sys.path.append(os.path.abspath(".."))
project_root = Path("..").resolve()
data_dir = project_root / "Data"

In [30]:
train_params = pd.read_csv("../Data/train_cosmology.csv")
train_halos = torch.tensor(np.load("../Data/train_halos.npy"))[..., :3] / 1000

test_params = pd.read_csv("../Data/test_cosmology.csv")
test_halos = torch.tensor(np.load("../Data/test_halos.npy"))[..., :3] / 1000
prior = torch.rand_like(train_halos)
prior_ot = torch.tensor(np.load("../Data/train_x0_ot.npy"))

In [31]:
print(f''' 
Shape of train_halos: {train_halos.shape}
Shape of test_halos: {test_halos.shape}
Shape of prior: {prior.shape}
Shape of prior_ot: {prior_ot.shape}
''')

 
Shape of train_halos: torch.Size([1800, 5000, 3])
Shape of test_halos: torch.Size([200, 5000, 3])
Shape of prior: torch.Size([1800, 5000, 3])
Shape of prior_ot: torch.Size([1800, 5000, 3])



In [32]:
for i in range(100):
    diffs = []
    diffs_ot = []
    prior = torch.rand_like(train_halos)

    for cosmology in range(train_halos.shape[0]):
        diffs.append(np.abs(min_image(train_halos[cosmology] - prior[cosmology], **BOX)))
        diffs_ot.append(np.abs(min_image(train_halos[cosmology] - prior_ot[cosmology], **BOX)))
    mean_diff = np.mean(diffs)
    mean_diff_ot = np.mean(diffs_ot)
    print(f"mean_diff: {mean_diff} ------- mean_diff_ot: {mean_diff_ot}")
    if mean_diff_ot > mean_diff:
        print("ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!")

/scratch-local/bartb.21342679/ipykernel_2596879/174096226.py:7: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  diffs.append(np.abs(min_image(train_halos[cosmology] - prior[cosmology], **BOX)))
/scratch-local/bartb.21342679/ipykernel_2596879/174096226.py:8: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  diffs_ot.append(np.abs(min_image(train_halos[cosmology] - prior_ot[cosmology], **BOX)))


mean_diff: 0.2500244677066803 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.2500024735927582 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.25000715255737305 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.25005215406417847 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.24998389184474945 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.2499965876340866 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.2500227391719818 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.2500039041042328 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.24999214708805084 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.2499963492155075 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.25000080466270447 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.2499759942293167 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.2500091791152954 ------- mean_diff_ot: 0.025722280144691467
mean_diff: 0.250027060508728 ------- mean_diff

KeyboardInterrupt: 

In [21]:
for i in range(100):
    diffs = []
    diffs_ot = []
    prior = torch.rand_like(test_halos)

    for cosmology in range(test_halos.shape[0]):
        diffs.append(np.abs(min_image(test_halos[cosmology] - prior[cosmology], **BOX)))
        diffs_ot.append(np.abs(min_image(test_halos[cosmology] - prior_ot[cosmology], **BOX)))
    mean_diff = np.mean(diffs)
    mean_diff_ot = np.mean(diffs_ot)
    print(f"mean_diff: {mean_diff} ------- mean_diff_ot: {mean_diff_ot}")
    if mean_diff_ot > mean_diff:
        print("ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!")

/scratch-local/bartb.21342679/ipykernel_2596879/2681544165.py:7: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  diffs.append(np.abs(min_image(test_halos[cosmology] - prior[cosmology], **BOX)))
/scratch-local/bartb.21342679/ipykernel_2596879/2681544165.py:8: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  diffs_ot.append(np.abs(min_image(test_halos[cosmology] - prior_ot[cosmology], **BOX)))


mean_diff: 0.2499152570962906 ------- mean_diff_ot: 0.25005701184272766
ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!
mean_diff: 0.25011295080184937 ------- mean_diff_ot: 0.25005701184272766
mean_diff: 0.2499345988035202 ------- mean_diff_ot: 0.25005701184272766
ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!
mean_diff: 0.2500365674495697 ------- mean_diff_ot: 0.25005701184272766
ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!
mean_diff: 0.24991099536418915 ------- mean_diff_ot: 0.25005701184272766
ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!
mean_diff: 0.2500346601009369 ------- mean_diff_ot: 0.25005701184272766
ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!
mean_diff: 0.24999487400054932 ------- mean_diff_ot: 0.25005701184272766
ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!
mean_diff: 0.2500293254852295 ------- mean_diff_ot: 0.25005701184272766
ERROR: OT DIFFERENCE IS BIGGER THAN UNIFORM DIFFERENCE!
mean_diff: 0.2499492466449737

In [7]:
print(np.mean(diffs), np.mean(diffs_ot))

0.25000855 0.250057


In [8]:
print(f'''
      shape of training parameters: {train_params.shape}
      shape of training halos: {train_halos.shape}
      shape of testing parameters: {test_params.shape}
      shape of testing halos: {test_halos.shape}
      ''')


      shape of training parameters: (1800, 5)
      shape of training halos: torch.Size([1800, 5000, 7])
      shape of testing parameters: (200, 5)
      shape of testing halos: torch.Size([200, 5000, 3])
      


In [9]:
train_params.nunique()

Omega_m    1800
Omega_b    1800
h          1800
n_s        1800
sigma_8    1800
dtype: int64

In [10]:
egnn = EGNN(t_embed_dim=config["model"]["fm"]["t_embed_dim"],
                input_node_d=config["model"]["egnn"]["input_node_d"],
                input_theta_d=5,
                theta_param_embd_dim=config["model"]["egnn"]["theta_param_embd_dim"],
                hidden_nf=config["model"]["egnn"]["hidden_nf"],
                latent_nf=config["model"]["egnn"]["latent_nf"],
                theta_nf=config["model"]["egnn"]["theta_nf"],
                n_layers=config["model"]["egnn"]["n_layers"],
                mlp_layers=config["model"]["egnn"]["mlp_layers"],
                single_layer=config["model"]["egnn"]["single_layer"],
                recurrent=config["model"]["egnn"]["recurrent"],
                activation=nn.SiLU(),
                norm=config["model"]["egnn"]["norm"],
                attention=config["model"]["egnn"]["attention"],
                scale_pred=config["model"]["egnn"]["scale_pred"],
                coords_weight=config["model"]["egnn"]["coords_weight"],
                norm_diff=config["model"]["egnn"]["norm_diff"])
    
model = FlowMatching(sigma_0=config["model"]["fm"]["sigma_0"],
                     sigma_sched=config["model"]["fm"]["sigma_sched"],
                     t_embed_dim=config["model"]["fm"]["t_embed_dim"],
                     version=config["model"]["fm"]["version"],
                     vnet=egnn,
                     batch_size=config["training"]["batch_size"],
                     prior=config["model"]["fm"]["prior"],
                     k=config["model"]["fm"]["k"],
                     t_embed=config["model"]["fm"]["t_embed"],
                     n_halos=config["model"]["fm"]["n_halos"],
                     dim=3
                     )

checkpoint = torch.load(f"/gpfs/home4/bartb/T5000/results/run_{train_run}/model_final.pth", map_location=torch.device('cpu'))
new_state_dict = {k.replace("module.", ""): v for k, v in checkpoint["model_state"].items()}  # Remove "module."
model.load_state_dict(new_state_dict)

print("Loaded model")

NameError: name 'config' is not defined

In [ ]:
train_halos.shape
masses = []

for cosmology in range(train_halos.shape[0]):
    for node in range(train_halos[cosmology].shape[0]):
        masses.append(train_halos[cosmology][node][-1])

print(np.mean(masses), np.var(masses))

In [ ]:
masses_log = torch.log10(torch.tensor(masses))
max_mass = torch.max(masses_log)
min_mass = torch.min(masses_log)
log_mass_norm = (masses_log - min_mass) / (max_mass - min_mass)
print(torch.mean(log_mass_norm), torch.std(log_mass_norm))
plt.hist(log_mass_norm, bins=100)


In [ ]:
# egnn.layers[0].node_mlp.mlp[0].bias